In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_2")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

condition_cycle_map = {}
for condition in condition_pal_map:
    for cycle in range(10, 15):
        condition_cycle_map[f"{condition}-{cycle}"] = condition_pal_map[condition][cycle-10]

## Mitotic Waves

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import pandas as pd
from scipy.spatial import KDTree

def estimate_surface_normal(delta_pos, weights):
    """PCA on weighted neighbor displacements → local surface normal."""
    W = np.diag(np.sqrt(weights))
    weighted = W @ delta_pos                        # (k, 3)
    _, _, Vt = np.linalg.svd(weighted, full_matrices=False)
    normal = Vt[-1]                                 # smallest singular value = normal
    return normal / np.linalg.norm(normal)

def estimate_gradients_tangent(df, k_neighbors=12, weight_decay=1.0):
    """
    Estimate gradient of 'corrected_division_time' constrained to the
    local tangent plane at each point.

    Returns df with: grad_x, grad_y, grad_z, grad_magnitude, nx, ny, nz
    """
    coords = df[['x', 'y', 'z']].values
    values = df['corrected_division_time'].values

    tree = KDTree(coords)
    dists, idxs = tree.query(coords, k=k_neighbors + 1)

    grad = np.zeros((len(df), 3))
    normals = np.zeros((len(df), 3))

    for i in range(len(coords)):
        neighbors = idxs[i, 1:]
        neighbor_dists = dists[i, 1:]

        delta_pos = coords[neighbors] - coords[i]   # (k, 3)
        delta_val = values[neighbors] - values[i]   # (k,)

        sigma = np.mean(neighbor_dists)
        weights = np.exp(-weight_decay * (neighbor_dists / sigma) ** 2)

        # --- Step 1: estimate local normal via PCA ---
        normal = estimate_surface_normal(delta_pos, weights)
        normals[i] = normal

        # --- Step 2: build orthonormal tangent basis (t1, t2) ---
        # Pick a vector not parallel to normal to cross with
        helper = np.array([1, 0, 0]) if abs(normal[0]) < 0.9 else np.array([0, 1, 0])
        t1 = np.cross(normal, helper)
        t1 /= np.linalg.norm(t1)
        t2 = np.cross(normal, t1)                   # already unit length

        # --- Step 3: project displacements into 2D tangent coords ---
        # Each neighbor displacement expressed as (u, v) in tangent plane
        A_2d = np.column_stack([delta_pos @ t1, delta_pos @ t2])  # (k, 2)

        W = np.diag(weights)
        AtWA = A_2d.T @ W @ A_2d                    # (2, 2)
        AtWb = A_2d.T @ (weights * delta_val)        # (2,)

        lam = 1e-6 * np.trace(AtWA) / 2
        g_2d, _, _, _ = np.linalg.lstsq(
            AtWA + lam * np.eye(2), AtWb, rcond=None
        )

        # --- Step 4: back-project to 3D ---
        grad[i] = g_2d[0] * t1 + g_2d[1] * t2

    df = df.copy()
    df['grad_x'] = grad[:, 0]
    df['grad_y'] = grad[:, 1]
    df['grad_z'] = grad[:, 2]
    df['grad_magnitude'] = np.linalg.norm(grad, axis=1)
    df['nx'] = normals[:, 0]
    df['ny'] = normals[:, 1]
    df['nz'] = normals[:, 2]
    return df

def get_division_gradient_cycle(df, cycle, k_neighbors=50):
    division_times_df = dnt.division_times.get_division_times(df)

    division_times_df_c12 = division_times_df.query("cycle == @cycle").copy()
    # remove outliers too far from the division time of their neighbors
    tree = KDTree(division_times_df_c12[["x", "y", "z"]].values)
    dists, idxs = tree.query(division_times_df_c12[["x", "y", "z"]].values, k=6)
    neighbor_times = division_times_df_c12["corrected_division_time"].values[idxs[:, 1:]]
    median_neighbor_time = np.median(neighbor_times, axis=1)
    time_diff = np.abs(division_times_df_c12["corrected_division_time"].values - median_neighbor_time)
    division_times_df_c12 = division_times_df_c12[time_diff < 1].copy()
    division_times_df_c12_20 = estimate_gradients_tangent(division_times_df_c12, k_neighbors=50)
    division_times_df_c12_50 = estimate_gradients_tangent(division_times_df_c12, k_neighbors=k_neighbors)

    g_20 = division_times_df_c12_20[["grad_x", "grad_y", "grad_z"]].values
    g_50 = division_times_df_c12_50[["grad_x", "grad_y", "grad_z"]].values

    print(((g_20 * g_50).sum(axis=1) / (np.linalg.norm(g_20, axis=1) * np.linalg.norm(g_50, axis=1))).mean())

    return division_times_df_c12_50

In [ ]:
def signed_distance_from_plane(points, plane_point, normal):
    """Signed distance of each point from a plane defined by a point + normal."""
    n = normal / np.linalg.norm(normal)
    return np.sum((points - plane_point) * n, axis=-1)

def opacity_from_distance(distances, cutoff=0.0, falloff=50.0, min_alpha=0.1):
    """
    Points beyond `cutoff` fade toward `min_alpha` over `falloff` units.
    Points at or in front of the plane stay fully opaque.
    """
    beyond = np.clip(distances - cutoff, 0, None)          # only positive = behind plane
    alpha = 1.0 - (1.0 - min_alpha) * np.clip(beyond / falloff, 0, 1)
    return alpha.astype(np.float32)

def update_point_opacity(points, pts_layer, colors, napari_viewer, plane_distance_from_camera=80.0, falloff=40.0, min_alpha=0.05):
    """
    Recompute per-point alpha based on current camera orientation.

    plane_distance_from_camera: how far in front of the camera centre the
                                 cutoff plane sits (world units)
    falloff:  distance over which opacity fades after the plane
    min_alpha: minimum opacity for far-away points
    """
    cam_center = np.array(napari_viewer.camera.center)
    normal = np.array(napari_viewer.camera.calculate_nd_view_direction(3, (0, 1, 2)))
    plane_point = cam_center + normal * plane_distance_from_camera

    points = pts_layer.data[..., :3]
    if len(points.shape) == 3:
        points = points[:, 0, :]

    points = np.array([pts_layer.data_to_world(p) for p in points])

    dists = signed_distance_from_plane(points, plane_point, normal)
    print(dists)
    alphas = opacity_from_distance(dists, cutoff=0.0, falloff=falloff, min_alpha=min_alpha)
    print(alphas.mean())

    # Build RGBA — reuse existing face colours, just overwrite alpha
    colors[:, 3] = alphas
    border_colors = np.zeros_like(colors)
    border_colors[:, 3] = alphas
    pts_layer.face_color = colors
    pts_layer.border_color = border_colors
    pts_layer.refresh()  # trigger napari redraw

In [ ]:
import napari

def do_napari_thing(k, cycle, angles, k_neighbors=50, factor=1300):

    print(stems[k])

    div_grads = get_division_gradient_cycle(spots_dfs[k], cycle=cycle, k_neighbors=k_neighbors)


    viewer = napari.Viewer(ndisplay=3)
    viewer.theme = "light"
    points = div_grads[["z", "y", "x"]].values
    vecs = div_grads[["grad_z", "grad_y", "grad_x"]].values * factor

    t = div_grads["corrected_division_time"].values

    colors = sns.husl_palette(l=0.6, as_cmap=True)(np.clip((t - (t.mean() - 1.5)) / 4, 0.0, 0.66))


    p = viewer.add_points(points, size=4, face_color=colors, border_color="w", opacity=1)
    vectors = viewer.add_vectors(np.stack([points, vecs], axis=1, ), edge_color=colors, edge_width=3, opacity=1)

    viewer.camera.angles = angles
    viewer.camera.zoom = 3.7

    update_point_opacity(points, vectors, vectors.edge_color, viewer, plane_distance_from_camera=14.0, falloff=100.0, min_alpha=0.05)
    update_point_opacity(points, p, p.face_color, viewer, plane_distance_from_camera=14.0, falloff=100.0, min_alpha=0.05)

    return viewer





In [ ]:
do_napari_thing(k=3, cycle=12, angles=(86, -15, 102), k_neighbors=200, factor=1300)

In [ ]:
do_napari_thing(k=6 , cycle=13, angles=(57, -85, -57), k_neighbors=50, factor=1300)

In [ ]:
viewer = do_napari_thing(k=8, cycle=12, angles=(90, -11, 93), k_neighbors=200, factor=1300)

In [ ]:
print(viewer.camera.angles)
print(viewer.camera.zoom)

In [ ]:
fig, ax = plt.subplots(1, 1)
division_times_df = dnt.division_times.get_division_times(spots_dfs[0])
sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="corrected_division_time", alpha=1.0, edgecolor="k")
# division_times_df["correction"] = division_times_df["corrected_division_time"] - division_times_df["division_time"]
# division_times_df["random_division_time"] = division_times_df["division_time"] + division_times_df["correction"].sample(frac=1).values + 2
# sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="random_division_time", color="r", alpha=0.5)
plt.ylim(28, 32)
plt.ylabel("Division time (minutes since start)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1)
division_times_df = dnt.division_times.get_division_times(spots_dfs[10])
print(stems[8])
sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="corrected_division_time", alpha=1.0, edgecolor="k")
# division_times_df["correction"] = division_times_df["corrected_division_time"] - division_times_df["division_time"]
# division_times_df["random_division_time"] = division_times_df["division_time"] + division_times_df["correction"].sample(frac=1).values + 2
# sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="random_division_time", color="r", alpha=0.5)
# plt.ylim(28, 32)
plt.ylabel("Division time (minutes since start)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

In [ ]:
division_times_df = dnt.division_times.get_division_times(spots_dfs[3])
sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="corrected_division_time", alpha=0.5)
# division_times_df["correction"] = division_times_df["corrected_division_time"] - division_times_df["division_time"]
# division_times_df["random_division_time"] = division_times_df["division_time"] + division_times_df["correction"].sample(frac=1).values + 2
# sns.scatterplot(division_times_df.query("cycle == 12 and AP.between(0.0, 0.95)"), x="AP", y="random_division_time", color="r", alpha=0.5)
# plt.ylim(30, 34)
plt.ylabel("Division time (minutes since start)")
plt.show()

## Pseudotime progression

In [ ]:

cycle = 12

for region in [(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]:
    df = spots_dfs[9]

    df = df.query("cycle == @cycle and distance < 3 and AP.between(@region[0], @region[1])").copy()

    df["time_since_cycle_start"] = df["time_since_nc11"] - df.groupby("tracklet_id")["time_since_nc11"].transform("min")
    df["cut_pseudotime"] = (df["pseudotime"] // 0.01) * 0.01

    sns.lineplot(x="cut_pseudotime", y="time_since_cycle_start", data=df, errorbar="sd")
plt.show()

## Nuclear cycle timing

In [ ]:
from collections import defaultdict

frame_df = defaultdict(list)

for k, df in enumerate(spots_dfs):
    condition = condition_map[stems[k][:-6]]

    g = df.groupby("frame")
    f_cycle = g["cycle"].agg(lambda x: x.mode()[0])
    f_time = g["dt"].agg(lambda x: x.fillna(0).mode())
    frame_df["cycle"].extend(f_cycle)
    frame_df["frame"].extend(f_cycle.index)
    frame_df["condition"].extend([condition] * len(f_cycle))
    frame_df["time"].extend(f_time)
    frame_df["source"].extend([k] * len(f_cycle))
    frame_df["condition-cycle"].extend([f"{condition}-{cycle}" for cycle in f_cycle])

frame_df = pd.DataFrame(frame_df)

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
for cycle in [13, 12, 11]:
    filtered = frame_df.query("cycle > 10 and cycle <= @cycle")
    length = filtered.groupby("source")["time"].sum()
    condition = filtered.groupby("source")["condition"].first()
    source = condition.index
    pal = {c: condition_cycle_map[f"{c}-{cycle}"] for c in condition_main_colors}
    sns.barplot(x=source, y=length, hue=condition, palette=pal, ax=ax, legend=False)

plt.show()


### Nuclear cycle Movement

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(6, 5), sharex=True, sharey=True)

for cycle, ax in zip([11, 12, 13], axes):

    for k, df in enumerate(spots_dfs[:3]):
        condition = "wt"
        if condition_map[stems[k][:-6]] != condition:
            continue
        df = df.query("AP.between(0.2, 0.8)")

        time_window = (-6, 10)

        track_df = df.groupby(["track_id", "frame"]).agg({
            "AP": "mean",
            "pseudotime": "mean",
            "dAP": "mean",
            "cycle": "mean",
            "distance": "mean",
            "time_since_nc11": "mean",
            "tracklet_id": "nunique",
            "x": "mean",
            "y": "mean",
            "z": "mean",
        }).reset_index()

        for col in ["x", "y", "z", "time_since_nc11"]:
            track_df[col] = track_df.groupby("track_id")[col].transform(lambda x: x.rolling(5, center=True, min_periods=1).mean())
            track_df[f"d_{col}"] = track_df.groupby("track_id")[col].diff()

        track_df["speed"] = np.sqrt(track_df["d_x"]**2 + track_df["d_y"]**2 + track_df["d_z"]**2) / track_df["d_time_since_nc11"]

        t = track_df.groupby("track_id")["tracklet_id"].max()
        t = t[t > 14]
        track_df = track_df[track_df["track_id"].isin(t.index)].copy()

        track_first_cycle_time = track_df.query("cycle > @cycle - 0.6 and cycle < @cycle + 0.1").groupby("track_id")["time_since_nc11"].min()

        track_df["time_since_cycle"] = track_df["time_since_nc11"] - track_df["track_id"].map(track_first_cycle_time)
        track_df["time_since_cycle"] = (track_df["time_since_cycle"] // 0.33) * 0.33

        sns.lineplot(track_df.query("time_since_cycle > @time_window[0] and time_since_cycle < @time_window[1]"), x="time_since_cycle", y="speed", legend=False, ax=ax, color=condition_pal_map[condition][cycle-10], lw=2, errorbar=None)

    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_ylim(0, 8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.axvline(0, color="k", ls="--")

fig.supylabel("Speed (µm/min)")
fig.supxlabel("Time since mitosis")
plt.show()



In [ ]:
for cycle in [11, 12, 13]:
    for k, df in enumerate(spots_dfs):
        if stems[k] == "20250705_spots":
            continue

        df = df[df["cycle"] == cycle].copy()
        df = df[df["distance"] < 3]
        df = df[df["pseudotime"] > 0.04]
        df = df[df["AP"].between(0.05, 0.95)]
        df["cut_pseudotime"] = (df["pseudotime"] // 0.01) * 0.01
        x = (df["pseudotime"] // 0.01) * 0.01
        y = df["dAP"]

        print(f"cycle: {cycle}, condition: {condition_map[stems[k][:-6]]}, dAP: {df.groupby("cut_pseudotime")["dAP"].mean().sum()}")

        condition = condition_map[stems[k][:-6]]

        if condition != "wt":
            continue
        # if condition == "bcd":
        #     continue

        sns.lineplot(x=x, y=y, color=condition_main_colors[condition], alpha=1, errorbar=None, label=f"{stems[k][:-6]}", legend=False, lw=2)

    plt.title(f"NC {cycle}")
    plt.xlabel("Pseudotime")
    plt.ylabel("dAP/dt (embryo lengths per minute)")
    plt.savefig(save_path / f"nc_{cycle}_dAP_pseudotime_allconditions.png", dpi=300, bbox_inches="tight")
    plt.show()